# 4. Sequential recorders

In sequential recorders such as DNA Typewriter, the target sites of a
cassette are edited in a fixed order: the second site can only be written
after the first, and so on. The order carries information about time that a
Cas9 model, which treats every site independently, ignores.

This notebook fits a dataset simulated with Cassiopeia's sequential
lineage-tracing simulator (250 cells, 12 cassettes of 5 sites, some missing
data) with the Cas9 model and with `SequentialModel`, and compares both with
neighbour joining. It takes about ten minutes on a laptop CPU.

In [1]:
import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from heracles import Heracles, robinson_foulds, sokal_collapse

# HERACLES logs every fit to stderr and to heracles.log in the output folder.
# Keep only warnings here so the notebook stays readable.
logging.getLogger("heracles").setLevel(logging.WARNING)
warnings.formatwarning = lambda message, category, *args, **kwargs: f"{category.__name__}: {message}\n"

DATA = Path("data")
RUNS = Path("runs")
RUNS.mkdir(exist_ok=True)
# Tutorial 1 explains this warning; hide it here.
warnings.filterwarnings("ignore", message="the loss was still improving")

from heracles import Cas9Model, SequentialModel

`rf_to_truth` collapses a tree and the true tree the same way and returns
their Robinson-Foulds distance.

In [2]:
def rf_to_truth(tree, cm, truth_path):
    """RF distance (count, normalised) to the true tree, both Camin-Sokal collapsed."""
    truth = sokal_collapse(Path(truth_path).read_text(), cm)
    tree = sokal_collapse(tree, cm)
    return robinson_foulds(tree, truth), robinson_foulds(tree, truth, normalized=True)

## The data

Columns are grouped by cassette: `c3_2` is the second site of cassette 3.
`SequentialModel` needs the columns in this order, with the number of
columns a multiple of the cassette size. Within a cassette, edits fill the
sites from left to right, so an unedited site is never followed by an
edited one.

In [3]:
cm = pd.read_csv(DATA / "typewriter" / "characters.csv", index_col=0)
print(f"{cm.shape[0]} cells x {cm.shape[1]} sites, {(cm == -1).to_numpy().mean():.0%} missing")

cassettes = cm.to_numpy().reshape(len(cm), -1, 5)
observed = (cassettes != -1).all(axis=2)
edited = cassettes > 0
in_order = (edited[:, :, :-1] | ~edited[:, :, 1:]).all(axis=2)
print(f"fully observed cassettes edited left to right: {in_order[observed].mean():.1%}")
cm.iloc[:5, :15]

250 cells x 60 sites, 12% missing
fully observed cassettes edited left to right: 100.0%


,c1_1,c1_2,c1_3,c1_4,c1_5,c2_1,c2_2,c2_3,c2_4,c2_5,c3_1,c3_2,c3_3,c3_4,c3_5
cell,,,,,,,,,,,,,,,
cell001,28,0,0,0,0,11,4,21,0,0,26,0,0,0,0
cell002,28,0,0,0,0,11,4,21,0,0,26,28,0,0,0
cell003,20,7,2,14,26,14,9,19,0,0,5,1,8,0,0
cell004,20,18,18,21,0,20,0,0,0,0,14,9,0,0,0
cell005,20,7,1,28,14,14,9,0,0,0,5,0,0,0,0


Fit the same data with the Cas9 and the sequential edit models, and score the RapidNJ tree the Cas9 fit started from as a baseline.

In [4]:
truth_path = DATA / "typewriter" / "true_tree.nwk"
rows = []
models = {}
for name, edit_model in [("Cas9Model", Cas9Model()),
                         ("SequentialModel", SequentialModel(cassette_size=5))]:
    models[name] = Heracles(outdir=RUNS / name, edit_model=edit_model, seed=0)
    models[name].fit(cm)
    rf, nrf = rf_to_truth(models[name].trees_["nj"], cm, truth_path)
    rows.append({"method": f"HERACLES, {name}", "RF": rf, "normalised RF": round(nrf, 3)})

nj = models["Cas9Model"].diagnostics_["init_tree_newick"]
rf, nrf = rf_to_truth(nj, cm, truth_path)
rows.append({"method": "NJ (starting tree)", "RF": rf, "normalised RF": round(nrf, 3)})
pd.DataFrame(rows).set_index("method")

,RF,normalised RF
method,,
"HERACLES, Cas9Model",58,0.171
"HERACLES, SequentialModel",42,0.126
NJ (starting tree),65,0.192


`SequentialModel` also takes `initiation_ratio`, the ratio of the rate of
the first edit in a cassette to the rate of later edits (1 by default), and
`learn_initiation_ratio=True` to fit it. From the command line:

```bash
heracles fit characters.csv --outdir runs/typewriter --edit-model sequential --cassette-size 5
```